In [14]:
import re
import subprocess
from pathlib import Path

from memvid_sdk import create
import pymupdf4llm
from shared import embedder

def extract_text(pdf_path: Path) -> str:
    # pypdf mangles this PDF's font encoding (injects spurious mid-word
    # spaces) and pymupdf can't even parse it ("invalid key in dict"); the
    # system's poppler (pdftotext) extracts it cleanly, so shell out to it.
    result = subprocess.run(
        ["pdftotext", str(pdf_path), "-"],
        capture_output=True,
        check=True,
    )
    return result.stdout.decode("utf-8")


In [45]:
document = Path("./documents/Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf")
import pymupdf


page_count = pymupdf.open(document).page_count
doc = pymupdf4llm.to_text(document, pages=list(range(2, page_count)))

In [ ]:
data = doc.split("\n\n")


['1. Добавление МЧД ',
 'Машиночитаемая доверенность (далее МЧД) - цифровой аналог бумажной доверенности на подписание документов в электронном виде. ',
 'На Портале поставщиков существует два способа добавления доверенности в профиль пользователя: ',
 '\uf02d Профиль пользователя: «Операции с МЧД» - пользователь сам добавляет xml-файл доверенности к профилю пользователя, полученный и подписанный ЭП руководителя организации; ',
 '\uf02d Профиль компании: «Операции с МЧД» - у администратора есть возможность добавлять xml-файл доверенности к профилю пользователя сотрудников своей организации. ',
 '1.1 Профиль пользователя «Операции с МЧД» ',
 'Для добавления МЧД пользователю необходимо нажать на кнопку «Операции с МЧД» (Рисунок 1(5)) после чего откроется страница «Машиночитаемые доверенности пользователя» (Рисунок 2). ',
 'Рисунок 1 – Профиль пользователя ',
 'Рисунок 2 – Добавление МЧД ',
 'На открывшейся странице пользователю необходимо загрузить доверенность, добавив xml-файл в поле «

In [42]:

# for page in doc[2:]:
#     print(page.get_text())

items = [d['text'] for d in doc]
STORE_PATH = "knowledge.mv2"
embedder.embed_documents([items[0]])
# embeddings = [
#     [float(x) for x in vec]
#     for vec in embedder.embed_documents([item for item in items])
# ]

# Path("STORE_PATH").unlink(missing_ok=True)
# mem = create(STORE_PATH, enable_lex=True, enable_vec=True)
# # put_many(embeddings=[...]) in memvid-sdk 2.0.160 only persists the vector
# # index correctly for the LAST item when given a multi-item batch (all
# # earlier items' vectors are silently dropped/overwritten). Calling it once
# # per item avoids this and keeps every frame's vector correctly indexed.
# for item, embedding in zip(items, embeddings):
#     mem.put_many([item], embeddings=[embedding])

RuntimeError: Ollama API error: 500 {"error":"the input length exceeds the context length"}